# Cache-Augmented Generation

### Environment Setup

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

MODEL_NAME = os.getenv("MODEL_NAME")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

### Load the knowledge and build the system prompt

In [2]:
def extract_knowledge(path_to_file: str) -> str:
    with open(path_to_file, "r") as f:
        return f.read()
    
knowledge_base = ['../reports/final_report.md', '../README.md']

texts = [extract_knowledge(path) for path in knowledge_base]
knowledge = "\n\n".join(texts)

system_prompt = f"""
You are an assistant who provides concise factual answers on the capstone project of Jose Fernando Gonzales.
Context:
{knowledge}
Question:
""".strip()

In [3]:
# Count the tokens in the system prompt and the question

from openai import OpenAI

client = OpenAI()

response = client.responses.input_tokens.count(
    model=MODEL_NAME,
    instructions=system_prompt,
    input="What is the capstone about?"
)
print(response.input_tokens)

11299


### Create the agent

In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

model = init_chat_model(
    model=MODEL_NAME,
    max_tokens=1000
)

agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[],
    checkpointer=checkpointer
)

### Ask Questions

In [ ]:


config = {"configurable": {"thread_id": "2"}}

question1 = "What is the capstone about?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question1}
    ]
}, config=config)

print(response['messages'])

[HumanMessage(content='What is the capstone about?', additional_kwargs={}, response_metadata={}, id='94090968-1237-484e-afa7-98a91bee069e'), AIMessage(content='Jose Fernando Gonzales’ capstone project is about **predicting whether diabetic patients will be readmitted to a hospital within 30 days after discharge**.\n\n- It’s framed as a **binary classification** problem (`<30` days vs. `NO` / `>30`).\n- The pipeline emphasizes **high recall** (catching high-risk patients), plus **fairness checks** (especially across race) and **explainability** (SHAP).\n- The best-performing approach was a **Random Forest using PCA-reduced engineered features**, with a chosen operating threshold focused on recall.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 11299, 'total_tokens': 11422, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_

In [ ]:
question2 = "What was my last question?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question2}
    ]
}, config=config)

print(response['messages'][-1])

content='Your last question was: **“What was my last question?”**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 11730, 'total_tokens': 11747, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 11008}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DOPGw7xRwLQnjo0TudKLNd4Hpywbb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d34ec-3957-7a83-bd26-d90ef504b2aa-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11730, 'output_tokens': 17, 'total_tokens': 11747, 'input_token_details': {'audio': 0, 'cache_read': 11008}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [14]:
question2 = "Who is Jose Fernando Gonzales??"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question2}
    ]
}, config=config)

message = response['messages'][-1].content

print(message)

**Jose Fernando Gonzales** is the **author** of the capstone project described in your report: *“Predicting 30-Day Hospital Readmission for Diabetic Patients”* (a machine learning capstone).


In [15]:
question2 = "How did he do the data cleaning?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question2}
    ]
}, config=config)

message = response['messages'][-1].content

print(message)

He did data cleaning in these main ways:

- **Handled missing values marked as `?`:** recoded `?` entries to missing values, then applied the planned feature handling.
- **Dropped unusable or very sparse/low-value fields:**
  - Removed identifiers (`encounter_id`, `patient_nbr`) for modeling.
  - Dropped **`weight`** (very high missingness).
  - Dropped **`payer_code`** (high missingness / weak value).
  - Dropped constant medication columns (`examide`, `citoglipton`).
- **Imputed/recoded categorical missingness:**
  - **`race` missing → `Unknown`**
  - **`medical_specialty` missing or rare categories → grouped** into `Other` / `Unknown`
- **Eliminated “structurally impossible” readmissions (leakage filtering):** removed encounters whose discharge dispositions make 30-day readmission impossible (discharge codes `{11, 17, 19, 20, 27}`), dropping **1,671** rows before modeling.
- **Standardized types and removed obvious artifacts** (e.g., correct coding for administrative variables) to m